# 167 — Component Contribution Analysis

Measure each component's contribution to the final prediction:
cumulative logit buildup, norm contributions, direction alignment,
interference patterns, and importance ranking.

## Why This Matters

This module provides tools for analyzing model internals. Understanding the internal mechanisms of transformer models is the core goal of mechanistic interpretability research — enabling us to move from treating models as black boxes to understanding the algorithms they implement.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.component_contribution_analysis import (
    cumulative_component_contribution,
    component_norm_contribution,
    component_direction_alignment,
    component_interference,
    component_importance_ranking,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.arange(15)

In [ ]:
result = cumulative_component_contribution(model, tokens)
print(f"Target token: {result['target_token']}  Final logit: {result['final_logit']:.4f}\n")
for c in result['per_component']:
    bar = '#' * int(abs(c['logit_contribution']) * 20)
    sign = '+' if c['logit_contribution'] > 0 else '-'
    print(f"  {c['component']:10s}: {sign}{abs(c['logit_contribution']):.4f}  "
          f"cumulative={c['cumulative_logit']:.4f}  {sign}{bar}")

In [ ]:
result = component_norm_contribution(model, tokens)
print(f"Final residual norm: {result['final_residual_norm']:.4f}\n")
for c in result['per_component']:
    bar = '#' * int(c['fraction_of_residual'] * 30)
    print(f"  {c['component']:10s}: norm={c['output_norm']:.4f}  "
          f"frac={c['fraction_of_residual']:.3f}  {bar}")

In [ ]:
result = component_direction_alignment(model, tokens)
for c in result['per_component']:
    bar = '#' * int(abs(c['alignment']) * 30)
    sign = '+' if c['alignment'] > 0 else '-'
    print(f"  {c['component']:10s}: alignment={c['alignment']:+.4f}  {sign}{bar}")

In [ ]:
result = component_interference(model, tokens)
for p in result['per_pair']:
    print(f"  {p['component_i']:10s} x {p['component_j']:10s}: "
          f"cos={p['cosine']:+.4f}  {p['type']}")

In [ ]:
result = component_importance_ranking(model, tokens)
print(f"Target token: {result['target_token']}\n")
for c in result['ranked_components']:
    bar = '#' * int(c['abs_contribution'] * 20)
    print(f"  #{c['rank']}: {c['component']:10s}  logit={c['logit_contribution']:+.4f}  {bar}")